# Atividade 3: YOLO11 no dataset Thesis (resíduos recicláveis)

Dataset: <https://universe.roboflow.com/ed-mikael-bebic-iyvku/thesis-efjq4> (versão 31, 4 classes, CC BY 4.0).

Antes de rodar: ativar a GPU (T4) e criar ROBOFLOW_API_KEY nos "Secrets" do Colab com "Acesso ao notebook" ligado.

In [ ]:
!pip install -q ultralytics roboflow

import ultralytics
ultralytics.checks()  # confirma versão e GPU disponível

In [ ]:
# Baixa o dataset no formato YOLOv8 (a chave vem dos Secrets do Colab, não fica no código)
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("ed-mikael-bebic-iyvku").project("thesis-efjq4")
dataset = project.version(31).download("yolov8", location="/content/dataset")

In [ ]:
# Inspeciona o dataset: data.yaml e contagem de imagens e labels
import os, glob

root = "/content/dataset"
print(open(f"{root}/data.yaml").read())

for split in ["train", "valid", "test"]:
    imgs = glob.glob(f"{root}/{split}/images/*")
    lbls = glob.glob(f"{root}/{split}/labels/*")
    print(split, "-> imagens:", len(imgs), "| labels:", len(lbls))

In [ ]:
# Corrige os caminhos do data.yaml para absolutos (evita erro de pasta não encontrada)
import yaml

cfg = yaml.safe_load(open(f"{root}/data.yaml"))
cfg["train"] = f"{root}/train/images"
cfg["val"] = f"{root}/valid/images"
cfg["test"] = f"{root}/test/images"
yaml.safe_dump(cfg, open(f"{root}/data.yaml", "w"))
print(open(f"{root}/data.yaml").read())

In [ ]:
# 4 imagens de treino, cada uma com o seu label: classe x_centro y_centro largura altura (0 a 1)
import random
from IPython.display import Image, display

amostras = random.sample(glob.glob(f"{root}/train/images/*"), 4)
for p in amostras:
    display(Image(p, width=300))
    label = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    for linha in open(label).read().splitlines():
        c, *coords = linha.split()
        print(c, f"({cfg['names'][int(c)]})", *coords)

In [ ]:
# Treina a partir dos pesos pré-treinados (transfer learning)
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
results = model.train(
    data=f"{root}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="/content/runs",
    name="thesis",
)

In [ ]:
# Avalia no conjunto de validação e mostra os gráficos
metrics = model.val()
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

display(Image("/content/runs/thesis/results.png", width=900))
display(Image("/content/runs/thesis/confusion_matrix.png", width=600))

## Uso do modelo treinado

O treino gera o arquivo `best.pt`. O `uso.py` é um programa separado do treino: precisa só da biblioteca
`ultralytics` e do `best.pt`, sem dataset nem GPU, e por isso roda em qualquer computador.
Aqui ele é usado em 3 imagens do conjunto de teste, que o modelo não viu no treino.

Para testar com outras imagens, envie-as através dos "Arquivos" do Colab e rode `!python uso.py foto.jpg`.

In [ ]:
# Baixa o uso.py do repositório e coloca o modelo treinado na mesma pasta
!wget -q -O uso.py https://raw.githubusercontent.com/data-science-unicamp/aplicacoes-data-science/main/atividade3/uso.py
!cp /content/runs/thesis/weights/best.pt .

testes = " ".join(f'"{p}"' for p in sorted(glob.glob(f"{root}/test/images/*"))[:3])
!python uso.py {testes}

for p in sorted(glob.glob("/content/runs/detect/predict*/*")):
    display(Image(p, width=400))

In [ ]:
# Baixa o best.pt, os gráficos e as imagens de teste num zip
from google.colab import files

!cd /content && zip -qr runs.zip runs
files.download("/content/runs.zip")